In [2]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import time
from datetime import datetime

import matplotlib.pyplot as plt
from ase import units
from ase.io import read
from ase.build import molecule
from ase.optimize import BFGS
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.geometry import get_distances
from ase.units import fs
from fairchem.core import pretrained_mlip, FAIRChemCalculator

W0420 02:24:03.059000 1640 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.


In [ ]:
# ── Calculator ────────────────────────────────────────────────────────────────
# initialize
predictor = pretrained_mlip.get_predict_unit('uma-s-1p1', device='cpu')
calc = FAIRChemCalculator(predictor, task_name='omat')

AssertionError: device must be either 'cpu' or 'cuda'

In [5]:
# ── Load MOF ──────────────────────────────────────────────────────────────────
mof_74 = read('mg_mof74.cif')

# ── Reference energies — trajectory-averaged NVT MD (thermal reference) ───────
# WHY: Static geometry-optimisation gives 0 K energies.  The production runs
# are NVT MD at finite T, so the correct reference is the *thermal* mean energy
# of (a) the bare MOF and (b) the isolated CO2 molecule each equilibrated at
# the same temperature.  Using 0 K references introduces a systematic offset
# (~5 eV) that grows with temperature.
#
# STRATEGY: run a short NVT MD for each reference system, discard the first
# 20 % as equilibration, then take <E_pot> over the production window.
# The function returns a dict  T -> E_ref  so per-temperature references are
# used when computing E_bind in the main sweep.

CO2_REF_STEPS     = 500    # steps per reference MD  (increase for better stats)
MOF_REF_STEPS     = 1000
REF_TIMESTEP  = 0.5   # fs
REF_TRAJ      = 10    # save every N steps
REF_EQ_FRAC   = 0.20  # fraction discarded as equilibration


def _md_mean_energy(atoms_in, temperature_K, n_steps, timestep_fs,
                    traj_every, eq_frac, label):
    """Run a short NVT MD and return the production-averaged potential energy."""
    atoms = atoms_in.copy()
    atoms.calc = calc

    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
    Stationary(atoms)

    dyn = NoseHooverChainNVT(
        atoms,
        timestep      = timestep_fs * fs,
        temperature_K = temperature_K,
        tdamp         = 100 * fs,
    )

    energies = []

    def _record():
        energies.append(atoms.get_potential_energy())

    dyn.attach(_record, interval=traj_every)
    dyn.run(steps=n_steps)

    eq_cut = max(1, int(len(energies) * eq_frac))
    mean_E = float(np.mean(energies[eq_cut:]))
    print(f'  {label} @ {temperature_K} K : '
          f'<E> = {mean_E:.6f} eV  '
          f'(prod frames = {len(energies) - eq_cut})')
    return mean_E


print('Computing trajectory-averaged reference energies ...')
print(f'  {len(list(range(300, 1050, 50)))} temperatures × 2 reference systems')

temperatures_ref = list(range(300, 1050, 50))

# ── Bare MOF reference ────────────────────────────────────────────────────────
mof_ref = mof_74.copy()
mof_ref.calc = calc
print('\nBare MOF references:')
E_mof_T = {}
for T in temperatures_ref:
    E_mof_T[T] = _md_mean_energy(
        mof_ref, T,
        n_steps=MOF_REF_STEPS, timestep_fs=REF_TIMESTEP,
        traj_every=REF_TRAJ, eq_frac=REF_EQ_FRAC,
        label='MOF'
    )

# ── Isolated CO2 reference ────────────────────────────────────────────────────
co2_base = molecule('CO2')
co2_base.set_pbc(False)
co2_base.center(vacuum=10.0)
co2_base.calc = calc
print('\nIsolated CO2 references:')
E_co2_T = {}
for T in temperatures_ref:
    E_co2_T[T] = _md_mean_energy(
        co2_base, T,
        n_steps=CO2_REF_STEPS, timestep_fs=REF_TIMESTEP,
        traj_every=REF_TRAJ, eq_frac=REF_EQ_FRAC,
        label='CO2'
    )

# Convenience single-temperature accessors used in pre-relax check below
# (use 300 K as the "cold" static-like reference for the pre-relaxation sanity check)
E_mof  = E_mof_T[300]
E_co2  = E_co2_T[300]

print(f'\nReference at 300 K:')
print(f'  E(MOF) = {E_mof:.6f} eV')
print(f'  E(CO2) = {E_co2:.6f} eV')


Computing trajectory-averaged reference energies ...
  15 temperatures × 2 reference systems

Bare MOF references:
  MOF @ 300 K : <E> = -1170.927377 eV  (prod frames = 81)
  MOF @ 350 K : <E> = -1169.929831 eV  (prod frames = 81)
  MOF @ 400 K : <E> = -1168.871917 eV  (prod frames = 81)
  MOF @ 450 K : <E> = -1167.828917 eV  (prod frames = 81)
  MOF @ 500 K : <E> = -1167.054063 eV  (prod frames = 81)
  MOF @ 550 K : <E> = -1166.128098 eV  (prod frames = 81)
  MOF @ 600 K : <E> = -1165.371719 eV  (prod frames = 81)
  MOF @ 650 K : <E> = -1164.211804 eV  (prod frames = 81)
  MOF @ 700 K : <E> = -1163.518253 eV  (prod frames = 81)
  MOF @ 750 K : <E> = -1162.268107 eV  (prod frames = 81)
  MOF @ 800 K : <E> = -1161.478469 eV  (prod frames = 81)
  MOF @ 850 K : <E> = -1160.795954 eV  (prod frames = 81)
  MOF @ 900 K : <E> = -1160.017901 eV  (prod frames = 81)
  MOF @ 950 K : <E> = -1158.307032 eV  (prod frames = 81)
  MOF @ 1000 K : <E> = -1157.488400 eV  (prod frames = 81)

Isolated CO2 

In [4]:
# ── Hardcoded reference energies (from previous run — kernel died) ─────────────
E_mof_T = {
    300:  -1170.927377,
    350:  -1169.929831,
    400:  -1168.871917,
    450:  -1167.828917,
    500:  -1167.054063,
    550:  -1166.128098,
    600:  -1165.371719,
    650:  -1164.211804,
    700:  -1163.518253,
    750:  -1162.268107,
    800:  -1161.478469,
    850:  -1160.795954,
    900:  -1160.017901,
    950:  -1158.307032,
    1000: -1157.488400,
}

E_co2_T = {
    300:  -22.522742,
    350:  -22.562976,
    400:  -22.524729,
    450:  -22.505545,
    500:  -22.508393,
    550:  -22.529646,
    600:  -22.446196,
    650:  -22.517010,
    700:  -22.488526,
    750:  -22.476538,
    800:  -22.561484,
    850:  -22.502157,
    900:  -22.446900,
    950:  -22.381456,
    1000: -22.426128,
}

E_mof  = E_mof_T[300]
E_co2  = E_co2_T[300]

print("Reference energies loaded from hardcoded values.")

Reference energies loaded from hardcoded values.


In [5]:
# ── Load pre-placed structure from CIF ───────────────────────────────────────
from ase.io import read

system_md = read('mof_co2_initial.cif')
system_md.set_pbc(True)
system_md.calc = calc

# Identify framework vs CO2 atoms
# CO2 is the last 3 atoms (C + 2 O) added in the GUI
n_framework = len(system_md) - 3
print(f'System: {len(system_md)} atoms ({n_framework} MOF + 3 CO2)')
print(f'Last 3 atoms (CO2): {[system_md[i].symbol for i in range(n_framework, len(system_md))]}')

# ── Pre-relax ─────────────────────────────────────────────────────────────────
print('Pre-relaxing...')
t0 = time.time()
BFGS(system_md, logfile='md_prerelax.log').run(fmax=0.001)
print(f'Done in {time.time()-t0:.1f} s')

E_bind_prerelax = system_md.get_potential_energy() - E_mof - E_co2
print(f'E_bind (pre-MD) = {E_bind_prerelax:.4f} eV')
if E_bind_prerelax > 0:
    raise ValueError('E_bind positive before MD — check CO2 placement in CIF.')

System: 165 atoms (162 MOF + 3 CO2)
Last 3 atoms (CO2): ['C', 'O', 'O']
Pre-relaxing...
Done in 1083.9 s
E_bind (pre-MD) = -7.1483 eV


In [10]:
import os
# ── MD runner ─────────────────────────────────────────────────────────────────
TRAJ_FOLDER   = 'trajectories_gui_corrected'
os.makedirs(TRAJ_FOLDER, exist_ok=True)

best_mg_idx   = 'gui'   # used in trajectory filename only
static_E_bind = system_md.get_potential_energy() - E_mof - E_co2  # 300 K ref

def run_md(system_in, temperature_K, n_steps=5000, timestep_fs=0.5,
           tdamp_fs=100, traj_every=10, log_every=50):
    """
    Run NVT MD at temperature_K and record E_bind relative to the
    trajectory-averaged reference energies at the *same* temperature.
    This removes the systematic 0 K vs finite-T offset.
    """
    atoms = system_in.copy()
    atoms.calc = calc

    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K)
    Stationary(atoms)

    ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
    traj_file = os.path.join(TRAJ_FOLDER, f'md_site{best_mg_idx}_{temperature_K}K_{ts}.traj')
    log_file  = os.path.join(TRAJ_FOLDER, f'md_site{best_mg_idx}_{temperature_K}K_{ts}.log')

    dyn = NoseHooverChainNVT(
        atoms,
        timestep      = timestep_fs * fs,
        temperature_K = temperature_K,
        tdamp         = tdamp_fs * fs,
        trajectory    = traj_file,
        logfile       = log_file,
    )

    # Per-temperature thermal references (fall back to 300 K if T not in dict)
    e_mof_ref = E_mof_T.get(temperature_K, E_mof_T[300])
    e_co2_ref = E_co2_T.get(temperature_K, E_co2_T[300])

    records = []

    def log_step():
        step  = dyn.get_number_of_steps()
        E_pot = atoms.get_potential_energy()
        E_b   = E_pot - e_mof_ref - e_co2_ref   # thermal reference
        records.append((step, E_pot, E_b))
        if step % log_every == 0:
            print(f'  step {step:4d} | T = {atoms.get_temperature():6.1f} K | '
                  f'E_pot = {E_pot:.4f} eV | E_bind = {E_b:.4f} eV')

    dyn.attach(log_step, interval=traj_every)

    print(f'\n{"="*55}')
    print(f'  MD  |  T = {temperature_K} K  |  {n_steps} steps  |  {len(atoms)} atoms')
    print(f'  ref: E_mof = {e_mof_ref:.4f} eV  E_co2 = {e_co2_ref:.4f} eV')
    print(f'{"="*55}')

    t_start = time.time()
    dyn.run(steps=n_steps)
    elapsed = time.time() - t_start

    print(f'\n  Done in {elapsed:.1f} s  ({elapsed/n_steps*1000:.0f} ms/step)')
    print(f'  Trajectory saved to {traj_file}')

    return atoms, records, traj_file


In [11]:
# ── Temperature sweep ─────────────────────────────────────────────────────────
temperatures = range(300, 1050, 50)

all_records   = {}
all_trajs     = {}
final_structs = {}

for T in temperatures:
    print(f'\nStarting MD at {T} K...')
    final_T, records_T, traj_T = run_md(
        system_md,            # always from pre-relaxed structure
        temperature_K=T
    )
    all_records[T]   = records_T
    all_trajs[T]     = traj_T
    final_structs[T] = final_T

print('\nAll temperatures complete!')


Starting MD at 300 K...

  MD  |  T = 300 K  |  5000 steps  |  165 atoms
  ref: E_mof = -1170.9274 eV  E_co2 = -22.5227 eV
  step    0 | T =  289.7 K | E_pot = -1200.5984 eV | E_bind = -7.1483 eV
  step   50 | T =  153.7 K | E_pot = -1197.6111 eV | E_bind = -4.1610 eV
  step  100 | T =  124.7 K | E_pot = -1196.7496 eV | E_bind = -3.2995 eV
  step  150 | T =  168.6 K | E_pot = -1197.4283 eV | E_bind = -3.9782 eV
  step  200 | T =  145.0 K | E_pot = -1196.6766 eV | E_bind = -3.2264 eV
  step  250 | T =  169.4 K | E_pot = -1196.8974 eV | E_bind = -3.4473 eV
  step  300 | T =  219.5 K | E_pot = -1197.6183 eV | E_bind = -4.1681 eV
  step  350 | T =  180.9 K | E_pot = -1196.4248 eV | E_bind = -2.9747 eV
  step  400 | T =  183.4 K | E_pot = -1196.1014 eV | E_bind = -2.6513 eV
  step  450 | T =  248.3 K | E_pot = -1197.0822 eV | E_bind = -3.6321 eV
  step  500 | T =  208.2 K | E_pot = -1195.8053 eV | E_bind = -2.3552 eV
  step  550 | T =  201.4 K | E_pot = -1195.2236 eV | E_bind = -1.7735 eV


In [8]:



# ── Summary statistics ────────────────────────────────────────────────────────
for T, records in sorted(all_records.items()):
    E_binds = np.array([r[2] for r in records])
    eq_cut  = len(E_binds) // 5
    prod    = E_binds[eq_cut:]
    print(f'\nT = {T} K  (production = {len(prod)} frames)')
    print(f'  <E_bind> = {prod.mean():.4f} eV')
    print(f'  std      = {prod.std():.4f} eV')
    print(f'  min/max  = {prod.min():.4f} / {prod.max():.4f} eV')


T = 300 K  (production = 81 frames)
  <E_bind> = -2.0159 eV
  std      = 1.1076 eV
  min/max  = -4.1681 / 0.5905 eV

T = 350 K  (production = 81 frames)
  <E_bind> = -1.8418 eV
  std      = 1.1709 eV
  min/max  = -3.8754 / 1.0374 eV

T = 400 K  (production = 81 frames)
  <E_bind> = -1.8924 eV
  std      = 1.3109 eV
  min/max  = -4.5429 / 1.0870 eV

T = 450 K  (production = 81 frames)
  <E_bind> = -2.0996 eV
  std      = 1.4269 eV
  min/max  = -5.0274 / 0.7327 eV

T = 500 K  (production = 81 frames)
  <E_bind> = -2.2064 eV
  std      = 1.6782 eV
  min/max  = -5.1232 / 1.6803 eV

T = 550 K  (production = 81 frames)
  <E_bind> = -2.2341 eV
  std      = 1.9031 eV
  min/max  = -5.9885 / 1.6866 eV

T = 600 K  (production = 81 frames)
  <E_bind> = -2.7425 eV
  std      = 2.1776 eV
  min/max  = -7.3945 / 1.4833 eV

T = 650 K  (production = 81 frames)
  <E_bind> = -2.3349 eV
  std      = 2.2448 eV
  min/max  = -6.5354 / 1.5431 eV

T = 700 K  (production = 81 frames)
  <E_bind> = -2.6450 eV
  s